In [ ]:
"""
intput: video file
output:
#  Data Adapter ：
pose_data_payload = {
    0: { # Frame 0
        "right_shoulder": {"x": 0.51, "y": 0.50, "z": 0.50},
        "right_elbow": {"x": 0.60, "y": 0.35, "z": 0.45},
        "right_wrist": {"x": 0.65, "y": 0.30, "z": 0.50},
        "right_hip": {"x": 0.50, "y": 0.80, "z": 0.50}
    },
    1: { # Frame 1
        "right_shoulder": {"x": 0.52, "y": 0.50, "z": 0.50},
        "right_elbow": {"x": 0.65, "y": 0.20, "z": 0.55},
        "right_wrist": {"x": 0.75, "y": 0.05, "z": 0.60},
        "right_hip": {"x": 0.51, "y": 0.80, "z": 0.50}
    },
    # ...
}
"""
from typing import Dict, Any
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

model_path = 'pose_landmarker_heavy.task'
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    min_pose_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    output_segmentation_masks=False
)
# Create a PoseLandmarker object
detector = vision.PoseLandmarker.create_from_options(options)

# init the pose data payload
pose_data_payload: Dict[int, Any] = {}

# Open the webcam
cap = cv2.VideoCapture("../data/raw_videos/test_clear.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
print(f"Frames per second: {fps}")
# Process the video frames
success, frame = cap.read()
if success:
    print(f"success: {success}, frame shape: {frame.shape}")
    # Convert the BGR image to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    print(f"converted successfully, frame_rgb shape: {frame_rgb.shape}")
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    print(f"mp_image created successfully, mp_image shape: {mp_image.width}x{mp_image.height}")
    detection_result = detector.detect_for_video(mp_image, timestamp_ms=0)
    # ensure the detection result contains pose landmarks
    if detection_result.pose_landmarks:
        print("Pose landmarks detected:")

        # Extract all the 33 points
        landmarks = detection_result.pose_landmarks[0]

        current_frame_data = {
            "right_shoulder": {"x": landmarks[12].x, "y": landmarks[12].y, "z": landmarks[12].z},
            "right_elbow": {"x": landmarks[14].x, "y": landmarks[14].y, "z": landmarks[14].z},
            "right_wrist": {"x": landmarks[16].x, "y": landmarks[16].y, "z": landmarks[16].z},
            "right_hip": {"x": landmarks[24].x, "y": landmarks[24].y, "z": landmarks[24].z},
        }
        pose_data_payload[0] = current_frame_data
        print("--- Frame 0 Pose Data ---")
        print(f"Right Shoulder: {current_frame_data['right_shoulder']}")
    else:
        print("No pose landmarks detected.")
else:
    print("Failed to read the video frame.")

cap.release()
detector.close()



In [ ]:
"""
intput: video file
output:
#  Data Adapter ：
pose_data_payload = {
    0: { # Frame 0
        "right_shoulder": {"x": 0.51, "y": 0.50, "z": 0.50},
        "right_elbow": {"x": 0.60, "y": 0.35, "z": 0.45},
        "right_wrist": {"x": 0.65, "y": 0.30, "z": 0.50},
        "right_hip": {"x": 0.50, "y": 0.80, "z": 0.50}
    },
    1: { # Frame 1
        "right_shoulder": {"x": 0.52, "y": 0.50, "z": 0.50},
        "right_elbow": {"x": 0.65, "y": 0.20, "z": 0.55},
        "right_wrist": {"x": 0.75, "y": 0.05, "z": 0.60},
        "right_hip": {"x": 0.51, "y": 0.80, "z": 0.50}
    },
    # ...
}
"""
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

model_path = 'pose_landmarker_heavy.task'
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    min_pose_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    output_segmentation_masks=False
)
# Create a PoseLandmarker object
detector = vision.PoseLandmarker.create_from_options(options)

# init the pose data payload
pose_data_payload = {}
frame_idx = 0
# Open the webcam
cap = cv2.VideoCapture("../data/raw_videos/test_clear.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
print(f"Frames per second: {fps}")
while True:
    # Process the video frames
    success, frame = cap.read()
    if not success:
        print("End of video reached or failed to read the video frame.")
        break

    # Convert the BGR image to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    # set up for timestamp in milliseconds for the current frame
    timestamp_ms = int((frame_idx / fps) * 1000)

    detection_result = detector.detect_for_video(mp_image, timestamp_ms=timestamp_ms)
    # ensure the detection result contains pose landmarks
    if detection_result.pose_landmarks:
        print("Pose landmarks detected:")

        # Extract all the 33 points
        # Note: detection_result.pose_landmarks is a list of PoseLandmarkList, where each PoseLandmarkList corresponds to a detected person in the frame. For simplicity, we will only consider the first detected person (if multiple people are detected).
        landmarks = detection_result.pose_landmarks[0]

        current_frame_data = {
            "right_shoulder": {"x": landmarks[12].x, "y": landmarks[12].y, "z": landmarks[12].z},
            "right_elbow": {"x": landmarks[14].x, "y": landmarks[14].y, "z": landmarks[14].z},
            "right_wrist": {"x": landmarks[16].x, "y": landmarks[16].y, "z": landmarks[16].z},
            "right_hip": {"x": landmarks[24].x, "y": landmarks[24].y, "z": landmarks[24].z},
        }
        pose_data_payload[frame_idx] = current_frame_data
        print(f"--- Frame {frame_idx} Pose Data ---")
        print(f"Right Shoulder: {current_frame_data['right_shoulder']}")
        print(f"Right Elbow: {current_frame_data['right_elbow']}")
        print(f"Right Wrist: {current_frame_data['right_wrist']}")
        print(f"Right Hip: {current_frame_data['right_hip']}")

    else:
        pose_data_payload[frame_idx] = None
        print("No pose landmarks detected.")
    frame_idx += 1

cap.release()
detector.close()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def calculate_joint_velocity(joint_name):
    velocities = []
    joint_x = []
    joint_y = []
    joint_z = []

    for i in pose_data_payload.keys():
        joint_x.append(pose_data_payload[i][f"{joint_name}"]["x"])
        joint_y.append(pose_data_payload[i][f"{joint_name}"]["y"])
        joint_z.append(pose_data_payload[i][f"{joint_name}"]["z"])

    for i in range(1, len(joint_x)):
        dx = joint_x[i] - joint_x[i - 1]
        dy = joint_y[i] - joint_y[i - 1]
        dz = joint_z[i] - joint_z[i - 1]
        velocity = np.sqrt(dx ** 2 + dy ** 2 + dz ** 2)
        velocities.append(velocity)
    return velocities


joint_name = ["right_shoulder", "right_elbow", "right_wrist", "right_hip"]
plt.figure()

for joint in joint_name:
    # Display the velocities

    plt.plot(calculate_joint_velocity(joint), label=f'{joint} Velocity')

plt.title(f'joint Velocity Over Time')
plt.xlabel('Frame')
plt.ylabel('Velocity')
plt.grid(True)
plt.show()




